# API Test — 20 URLs from with_label.csv

This notebook:
1. Reads 20 URLs from `app/data/with_label.csv`
2. Calls `POST /analyze` on the local API for each URL
3. Saves every response (success + errors) to `Notebook/results/api_test_results.json` and `api_test_results.csv`
4. Prints a summary table

**Pre-requisite:** API must be running locally.
```bash
cd app && uvicorn main:app --host 0.0.0.0 --port 8080
```

## 1. Configuration

In [12]:
# ── Configuration ─────────────────────────────────────────────────────────
API_BASE_URL   = "http://localhost:8080"   # change if running on a different host/port
CSV_PATH       = "../app/data/with_label.csv"
NUM_URLS       = 20                        # number of URLs to test
RANDOM_SAMPLE  = True                     # True = random sample, False = first N rows
RANDOM_SEED    = 42
REQUEST_TIMEOUT = 120                     # seconds per API call

# Output paths
import os
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
JSON_OUT = os.path.join(RESULTS_DIR, "api_test_results.json")
CSV_OUT  = os.path.join(RESULTS_DIR, "api_test_results.csv")

# API request payload template
FEATURES = {
    "categories": {"limit": 3},
    "keywords":   {"limit": 20},
    "entities":   {"limit": 10},
    "concepts":   {"limit": 10}
}

print(f"API base URL : {API_BASE_URL}")
print(f"CSV path     : {CSV_PATH}")
print(f"URLs to test : {NUM_URLS}")
print(f"Results dir  : {RESULTS_DIR}/")

API base URL : http://localhost:8080
CSV path     : ../app/data/with_label.csv
URLs to test : 20
Results dir  : results/


## 2. Install / import dependencies

In [13]:
# Install if missing
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "requests", "pandas"])

DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


0

In [14]:
import csv
import json
import time
import random
import requests
import pandas as pd
from datetime import datetime, timezone

csv.field_size_limit(2**20)  # raise limit for large text fields
print("Imports OK")

Imports OK


## 3. Health-check the API

In [15]:
try:
    resp = requests.get(f"{API_BASE_URL}/health", timeout=10)
    health = resp.json()
    print("Health check:", health)
    if not health.get("ready"):
        print("⚠️  Pipeline not ready — wait for startup to complete then re-run this cell.")
    else:
        print("✅ API is ready.")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to API at", API_BASE_URL)
    print("   Start it with: cd app && uvicorn main:app --host 0.0.0.0 --port 8080")

Health check: {'status': 'healthy', 'ready': True, 'model': 'mistralai/mistral-small-3-1-24b-instruct-2503'}
✅ API is ready.


## 4. Load URLs from CSV

In [16]:
# Read the CSV
df_source = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df_source)} rows, columns: {list(df_source.columns)}")

# Sample or slice
if RANDOM_SAMPLE:
    df_sample = df_source.sample(n=min(NUM_URLS, len(df_source)), random_state=RANDOM_SEED).reset_index(drop=True)
else:
    df_sample = df_source.head(NUM_URLS).reset_index(drop=True)

print(f"\nSelected {len(df_sample)} URLs for testing:")
for i, row in df_sample.iterrows():
    print(f"  {i+1:2d}. {row['url']}")

CSV loaded: 1153 rows, columns: ['url', 'text', 'label']

Selected 20 URLs for testing:
   1. https://www.managebt.org/webinars/
   2. https://www.britannica.com/topic/radio/A-new-art-form#ref301723
   3. https://www.merrickdodgechryslerjeep.com/vip-preowned-carefree-program.htm
   4. https://www.ffdistantworlds.com/music
   5. https://www.armis.com/solutions/full-asset-inventory-and-cmdb-enrichment/
   6. https://www.tackwarehouse.com/category-s/2883.htm
   7. https://pon.com/en/activities/automotive/wholesale-retail/pon-center/
   8. https://www.musicradar.com/news/page/7
   9. https://wordpress.com/abuse/?report_url=https://littlemaninmyhead.wordpress.com/2021/01/28/no-java-is-not-a-secure-programming-language/
  10. https://www.bmwrockville.com/new-inventory/index.htm?search=&model=X5 M&sortBy=year+desc
  11. https://www.skyhighsecurity.com/about/contact-us.html
  12. https://www.visitomaha.com/things-to-do/neighborhoods/little-bohemia/
  13. https://www.tandfeditingservices.com/se

## 5. Run the API calls

In [17]:
def call_analyze_api(url: str) -> dict:
    """
    Call POST /analyze and return a structured result dict.
    Always returns a dict — never raises; errors are captured in the result.
    """
    payload = {
        "url": url,
        "version": "2022-04-07",
        "return_analyzed_text": True,
        "features": FEATURES
    }
    t0 = time.perf_counter()
    try:
        resp = requests.post(
            f"{API_BASE_URL}/analyze",
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)

        if resp.status_code == 200:
            data = resp.json()
            return {
                "url":             url,
                "http_status":     resp.status_code,
                "call_status":     data.get("status", "unknown"),
                "latency_ms":      latency_ms,
                "categories":      data.get("categories", []),
                "keywords_theme":  data.get("keywords", {}).get("theme", ""),
                "keywords":        data.get("keywords", {}).get("keywords", []),
                "phrases":         data.get("keywords", {}).get("phrases", []),
                "entities":        data.get("entities", {}),
                "concepts":        data.get("concepts", []),
                "metadata":        data.get("metadata", {}),
                "analyzed_text":   (data.get("analyzed_text") or "")[:300],  # truncate for display
                "error":           data.get("error"),
                "error_code":      data.get("error_code"),
                "full_response":   data,
            }
        else:
            # Non-200 — capture the error detail
            latency_ms = round((time.perf_counter() - t0) * 1000, 1)
            try:
                err_body = resp.json()
                err_detail = err_body.get("detail", str(err_body))
                if isinstance(err_detail, dict):
                    error_msg  = err_detail.get("error", str(err_detail))
                    error_code = err_detail.get("error_code", "HTTP_ERROR")
                else:
                    error_msg  = str(err_detail)
                    error_code = f"HTTP_{resp.status_code}"
            except Exception:
                error_msg  = resp.text[:300]
                error_code = f"HTTP_{resp.status_code}"

            return {
                "url":           url,
                "http_status":   resp.status_code,
                "call_status":   "error",
                "latency_ms":    latency_ms,
                "categories":    [],
                "keywords_theme": "",
                "keywords":      [],
                "phrases":       [],
                "entities":      {},
                "concepts":      [],
                "metadata":      {},
                "analyzed_text": "",
                "error":         error_msg,
                "error_code":    error_code,
                "full_response": {"detail": error_msg},
            }

    except requests.exceptions.Timeout:
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "url": url, "http_status": None, "call_status": "error",
            "latency_ms": latency_ms, "categories": [], "keywords_theme": "",
            "keywords": [], "phrases": [], "entities": {}, "concepts": [],
            "metadata": {}, "analyzed_text": "",
            "error": f"Request timed out after {REQUEST_TIMEOUT}s",
            "error_code": "CLIENT_TIMEOUT", "full_response": {},
        }
    except requests.exceptions.ConnectionError as exc:
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "url": url, "http_status": None, "call_status": "error",
            "latency_ms": latency_ms, "categories": [], "keywords_theme": "",
            "keywords": [], "phrases": [], "entities": {}, "concepts": [],
            "metadata": {}, "analyzed_text": "",
            "error": f"Cannot connect to API: {exc}",
            "error_code": "API_CONNECTION_ERROR", "full_response": {},
        }

In [18]:
results = []
run_timestamp = datetime.now(timezone.utc).isoformat()

print(f"Starting {len(df_sample)} API calls at {run_timestamp}\n")

for i, row in df_sample.iterrows():
    url = row["url"]
    print(f"[{i+1:2d}/{len(df_sample)}] {url}")

    result = call_analyze_api(url)
    result["source_label"] = row.get("label", "")  # ground-truth label from CSV
    result["row_index"]    = int(i)
    result["tested_at"]    = datetime.now(timezone.utc).isoformat()
    results.append(result)

    # Print a compact one-line summary per URL
    if result["call_status"] == "success":
        cats = ", ".join(c["label"] for c in result["categories"][:2]) or "—"
        print(f"       ✅ {result['latency_ms']}ms | categories: {cats}")
    else:
        print(f"       ❌ {result['latency_ms']}ms | [{result['error_code']}] {str(result['error'])[:80]}")

print(f"\nDone. {sum(1 for r in results if r['call_status'] == 'success')} succeeded, "
      f"{sum(1 for r in results if r['call_status'] != 'success')} failed.")

Starting 20 API calls at 2026-10-23T10:37:32.820971+00:00

[ 1/20] https://www.managebt.org/webinars/
       ❌ 5177.5ms | [None] Failed to parse model response: Extra data: line 2 column 1 (char 345)
[ 2/20] https://www.britannica.com/topic/radio/A-new-art-form#ref301723
       ❌ 4245.8ms | [UNKNOWN] An unexpected error occurred while fetching the page. Please verify the URL and 
[ 3/20] https://www.merrickdodgechryslerjeep.com/vip-preowned-carefree-program.htm
       ❌ 4387.7ms | [UNKNOWN] An unexpected error occurred while fetching the page. Please verify the URL and 
[ 4/20] https://www.ffdistantworlds.com/music
       ✅ 4587.8ms | categories: /soundtracks, /concerts_&_music_festivals
[ 5/20] https://www.armis.com/solutions/full-asset-inventory-and-cmdb-enrichment/
       ❌ 3569.5ms | [UNKNOWN] An unexpected error occurred while fetching the page. Please verify the URL and 
[ 6/20] https://www.tackwarehouse.com/category-s/2883.htm
       ✅ 15062.5ms | categories: /equestrian_equipme

## 6. Save results

In [19]:
# ── Save full results as JSON (preserves all nested fields) ───────────────
json_output = {
    "run_timestamp": run_timestamp,
    "api_base_url":  API_BASE_URL,
    "total_tested":  len(results),
    "succeeded":     sum(1 for r in results if r["call_status"] == "success"),
    "failed":        sum(1 for r in results if r["call_status"] != "success"),
    "results":       [r["full_response"] | {
                          "url":          r["url"],
                          "http_status":  r["http_status"],
                          "latency_ms":   r["latency_ms"],
                          "source_label": r["source_label"],
                          "tested_at":    r["tested_at"],
                     } for r in results]
}

with open(JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(json_output, f, indent=2, ensure_ascii=False)

print(f"✅ JSON saved → {JSON_OUT}")

✅ JSON saved → results/api_test_results.json


In [20]:
# ── Save flat CSV (one row per URL, nested fields serialised as JSON strings) ─
csv_rows = []
for r in results:
    csv_rows.append({
        "row_index":       r["row_index"],
        "url":             r["url"],
        "http_status":     r["http_status"],
        "call_status":     r["call_status"],
        "latency_ms":      r["latency_ms"],
        "categories":      json.dumps(r["categories"],  ensure_ascii=False),
        "keywords_theme":  r["keywords_theme"],
        "keywords":        json.dumps(r["keywords"],    ensure_ascii=False),
        "phrases":         json.dumps(r["phrases"],     ensure_ascii=False),
        "concepts":        json.dumps(r["concepts"],    ensure_ascii=False),
        "entities_person": json.dumps(r["entities"].get("PERSON",       []), ensure_ascii=False),
        "entities_org":    json.dumps(r["entities"].get("ORGANIZATION", []), ensure_ascii=False),
        "entities_loc":    json.dumps(r["entities"].get("LOCATION",     []), ensure_ascii=False),
        "entities_date":   json.dumps(r["entities"].get("DATE",         []), ensure_ascii=False),
        "entities_event":  json.dumps(r["entities"].get("EVENT",        []), ensure_ascii=False),
        "entities_product":json.dumps(r["entities"].get("PRODUCT",      []), ensure_ascii=False),
        "entities_other":  json.dumps(r["entities"].get("OTHER",        []), ensure_ascii=False),
        "page_title":      r["metadata"].get("title",       ""),
        "page_domain":     r["metadata"].get("domain",      ""),
        "page_word_count": r["metadata"].get("word_count",   ""),
        "analyzed_text":   r["analyzed_text"],
        "error":           r.get("error") or "",
        "error_code":      r.get("error_code") or "",
        "source_label":    r["source_label"],
        "tested_at":       r["tested_at"],
    })

df_results = pd.DataFrame(csv_rows)
df_results.to_csv(CSV_OUT, index=False, encoding="utf-8")
print(f"✅ CSV  saved → {CSV_OUT}")
print(f"   Shape: {df_results.shape}")

✅ CSV  saved → results/api_test_results.csv
   Shape: (20, 25)


## 7. Summary table

In [21]:
import pandas as pd

summary_rows = []
for r in results:
    cats_str = " | ".join(
        f"{c['label']} ({c['score']})"
        for c in r["categories"]
    ) if r["categories"] else "—"

    summary_rows.append({
        "#":           r["row_index"] + 1,
        "URL":         r["url"][:60] + ("..." if len(r["url"]) > 60 else ""),
        "Status":      "✅" if r["call_status"] == "success" else "❌",
        "ms":          r["latency_ms"],
        "Categories":  cats_str[:80],
        "Theme":       r["keywords_theme"][:50],
        "Error":       (r.get("error") or "")[:60],
    })

df_summary = pd.DataFrame(summary_rows)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 30)
df_summary

,#,URL,Status,ms,Categories,Theme,Error
0,1,https://www.managebt.org/webinars/,❌,5177.5,—,,Failed to parse model response: Extra data: line 2 column 1
1,2,https://www.britannica.com/topic/radio/A-new-art-form#ref301...,❌,4245.8,—,,An unexpected error occurred while fetching the page. Please
2,3,https://www.merrickdodgechryslerjeep.com/vip-preowned-carefr...,❌,4387.7,—,,An unexpected error occurred while fetching the page. Please
3,4,https://www.ffdistantworlds.com/music,✅,4587.8,/soundtracks (0.9) | /concerts_&_music_festivals (0.7),Final Fantasy music,
4,5,https://www.armis.com/solutions/full-asset-inventory-and-cmd...,❌,3569.5,—,,An unexpected error occurred while fetching the page. Please
5,6,https://www.tackwarehouse.com/category-s/2883.htm,✅,15062.5,/equestrian_equipment_&_tack (0.9) | /footwear (0.8) | /clothing_accessories (0.,equestrian equipment and apparel,
6,7,https://pon.com/en/activities/automotive/wholesale-retail/po...,✅,5476.6,/automotive/wholesale_retail (0.9) | /volkswagen (0.8) | /audi (0.7),automotive retail services,
7,8,https://www.musicradar.com/news/page/7,✅,7473.2,/music_equipment_&_technology (0.9) | /music_reference (0.8) | /music_videos (0.,music news and equipment,
8,9,https://wordpress.com/abuse/?report_url=https://littlemaninm...,❌,4130.5,—,,An unexpected error occurred while fetching the page. Please
9,10,https://www.bmwrockville.com/new-inventory/index.htm?search=...,❌,4382.4,—,,An unexpected error occurred while fetching the page. Please


## 8. Stats

In [23]:
total      = len(results)
succeeded  = sum(1 for r in results if r["call_status"] == "success")
failed     = total - succeeded
latencies  = [r["latency_ms"] for r in results if r["call_status"] == "success"]
avg_ms     = round(sum(latencies) / len(latencies), 1) if latencies else 0
min_ms     = round(min(latencies), 1) if latencies else 0
max_ms     = round(max(latencies), 1) if latencies else 0

print("=" * 45)
print(f"  Total tested    : {total}")
print(f"  Succeeded       : {succeeded}  ({100*succeeded//total}%)")
print(f"  Failed          : {failed}")
print(f"  Avg latency     : {avg_ms} ms")
print(f"  Min latency     : {min_ms} ms")
print(f"  Max latency     : {max_ms} ms")
print("=" * 45)

# Error breakdown
if failed:
    print("\nError breakdown:")
    from collections import Counter
    err_counts = Counter(r["error_code"] for r in results if r["call_status"] != "success")
    for code, count in err_counts.most_common():
        print(f"  {code:<30} {count}")

# Category distribution
if succeeded:
    print("\nTop categories across all results:")
    from collections import Counter
    all_cats = [
        c["label"]
        for r in results if r["call_status"] == "success"
        for c in r["categories"]
    ]
    for label, count in Counter(all_cats).most_common(10):
        print(f"  {label:<50} {count}")

  Total tested    : 20
  Succeeded       : 5  (25%)
  Failed          : 15
  Avg latency     : 7329.7 ms
  Min latency     : 4048.5 ms
  Max latency     : 15062.5 ms

Error breakdown:
  UNKNOWN                        12


TypeError: unsupported format string passed to NoneType.__format__

## 9. Inspect a single result

In [24]:
# Change this index (0–19) to inspect any result in detail
IDX = 0

r = results[IDX]
print(f"URL    : {r['url']}")
print(f"Status : {r['call_status']}  ({r['latency_ms']} ms)")
print()
print("Categories:")
for c in r["categories"]:
    print(f"  {c['label']}  (score={c['score']})")
print()
print(f"Theme    : {r['keywords_theme']}")
print(f"Keywords : {r['keywords']}")
print(f"Phrases  : {r['phrases']}")
print(f"Concepts : {r['concepts']}")
print()
print("Entities:")
for etype, vals in r["entities"].items():
    if vals:
        print(f"  {etype:<15} {vals}")
print()
print("Metadata:", r["metadata"])
print()
print("Source label (ground truth):", r["source_label"])
if r.get("error"):
    print(f"\n⚠ Error [{r['error_code']}]: {r['error']}")

URL    : https://www.managebt.org/webinars/
Status : error  (5177.5 ms)

Categories:

Theme    : 
Keywords : []
Phrases  : []
Concepts : []

Entities:

Metadata: {}

Source label (ground truth): ['/computers_&_electronics/enterprise_technology', '/computers_&_electronics/software/business_&_productivity_software', '/business_&_industrial/business_services/consulting', '/business_&_industrial/business_operations/management', '/internet_&_telecom/teleconferencing', '/jobs_&_education/education/distance_learning']

⚠ Error [None]: Failed to parse model response: Extra data: line 2 column 1 (char 345)


## 10. Reload saved results (optional)

Run this cell at any time to reload previously saved results without re-running the API calls.

In [25]:
with open(JSON_OUT, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"Loaded {len(saved['results'])} results from {JSON_OUT}")
print(f"Run timestamp : {saved['run_timestamp']}")
print(f"Succeeded     : {saved['succeeded']} / {saved['total_tested']}")

Loaded 20 results from results/api_test_results.json
Run timestamp : 2026-10-23T10:37:32.820971+00:00
Succeeded     : 5 / 20
